In [2]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("data/anime.csv")

# Display first 5 rows
print("First 5 rows:")
display(df.head())

# Display dataset shape
print("\nDataset Shape:")
print(df.shape)

# Display column names
print("\nColumn Names:")
print(df.columns.tolist())

# Display data types
print("\nData Types:")
print(df.dtypes)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate Rows:")
print(df.duplicated().sum())

First 5 rows:


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266



Dataset Shape:
(12294, 7)

Column Names:
['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']

Data Types:
anime_id      int64
name            str
genre           str
type            str
episodes        str
rating      float64
members       int64
dtype: object

Missing Values:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Duplicate Rows:
0


In [3]:
# Check missing values
print("Missing values before preprocessing:")
print(df.isnull().sum())

# Fill missing genres
df['genre'] = df['genre'].fillna('Unknown')

# Fill missing types
df['type'] = df['type'].fillna('Unknown')

# Fill missing ratings with the median rating
df['rating'] = df['rating'].fillna(df['rating'].median())

# Convert episodes to numeric
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

# Fill missing episode values with median
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

# Check missing values after preprocessing
print("\nMissing values after preprocessing:")
print(df.isnull().sum())

# Check duplicates
print("\nNumber of duplicate rows:", df.duplicated().sum())

Missing values before preprocessing:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Missing values after preprocessing:
anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

Number of duplicate rows: 0


In [4]:
# Explore numerical features
print("Numerical feature statistics:")
display(df[['rating', 'episodes', 'members']].describe())

Numerical feature statistics:


,rating,episodes,members
count,12294.000000,12294.000000,1.229400e+04
mean,6.475700,12.095412,1.807134e+04
std,1.017179,46.244062,5.482068e+04
min,1.670000,1.000000,5.000000e+00
25%,5.900000,1.000000,2.250000e+02
50%,6.570000,2.000000,1.550000e+03
75%,7.170000,12.000000,9.437000e+03
max,10.000000,1818.000000,1.013917e+06


In [5]:
# Check unique values in important categorical columns
print("Number of unique genres:", df['genre'].nunique())
print("Number of anime types:", df['type'].nunique())

print("\nAnime types:")
print(df['type'].value_counts())

Number of unique genres: 3265
Number of anime types: 7

Anime types:
type
TV         3787
OVA        3311
Movie      2348
Special    1676
ONA         659
Music       488
Unknown      25
Name: count, dtype: int64


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(token_pattern=r'[^, ]+')

# Convert genre column into TF-IDF features
genre_matrix = tfidf.fit_transform(df['genre'])

print("Genre matrix shape:", genre_matrix.shape)
print("Number of genre features:", len(tfidf.get_feature_names_out()))

Genre matrix shape: (12294, 47)
Number of genre features: 47


In [8]:
from sklearn.preprocessing import StandardScaler

# Select numerical features
numerical_features = df[['rating', 'episodes', 'members']]

# Standardize numerical features
scaler = StandardScaler()
numerical_matrix = scaler.fit_transform(numerical_features)

print("Numerical matrix shape:", numerical_matrix.shape)

Numerical matrix shape: (12294, 3)


In [9]:
from scipy.sparse import hstack, csr_matrix

# Convert numerical features to sparse matrix
numerical_sparse = csr_matrix(numerical_matrix)

# Combine genre and numerical features
combined_features = hstack([genre_matrix, numerical_sparse])

print("Combined feature matrix shape:", combined_features.shape)

Combined feature matrix shape: (12294, 50)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate cosine similarity
cosine_sim = cosine_similarity(combined_features)

print("Cosine similarity matrix shape:", cosine_sim.shape)

Cosine similarity matrix shape: (12294, 12294)


In [12]:
def recommend_anime(anime_name, num_recommendations=10):
    # Find the index of the anime
    anime_indices = df[df['name'].str.lower() == anime_name.lower()].index

    if len(anime_indices) == 0:
        print("Anime not found in the dataset.")
        return

    anime_index = anime_indices[0]

    # Get similarity scores for the selected anime
    similarity_scores = cosine_sim[anime_index]

    # Sort anime by similarity score
    similar_indices = similarity_scores.argsort()[::-1]

    # Remove the selected anime itself
    similar_indices = similar_indices[similar_indices != anime_index]

    # Select top recommendations
    top_indices = similar_indices[:num_recommendations]

    # Create recommendation DataFrame
    recommendations = df.iloc[top_indices][
        ['name', 'genre', 'type', 'episodes', 'rating', 'members']
    ].copy()

    # Add similarity score
    recommendations['Similarity Score'] = similarity_scores[top_indices]

    return recommendations.reset_index(drop=True)

In [13]:
recommend_anime("Naruto", 10)

,name,genre,type,episodes,rating,members,Similarity Score
0,Fairy Tail,"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,175.0,8.22,584590,0.993032
1,D.Gray-man,"Action, Adventure, Comedy, Shounen",TV,103.0,8.20,334399,0.974623
2,Dragon Ball GT,"Action, Adventure, Comedy, Fantasy, Magic, Sci...",TV,64.0,6.72,226625,0.974601
3,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",TV,148.0,9.13,425855,0.973229
4,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",TV,153.0,8.16,316102,0.972025
5,Bleach,"Action, Comedy, Shounen, Super Power, Supernat...",TV,366.0,7.95,624055,0.967331
6,Soul Eater,"Action, Adventure, Comedy, Fantasy, Shounen, S...",TV,51.0,8.08,580184,0.957075
7,Fullmetal Alchemist,"Action, Adventure, Comedy, Drama, Fantasy, Mag...",TV,51.0,8.33,600384,0.955219
8,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,0.954936
9,Fairy Tail (2014),"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,102.0,8.25,255076,0.949190


In [14]:
def recommend_by_threshold(anime_name, threshold=0.5, max_recommendations=10):
    # Find the anime index
    anime_indices = df[df['name'].str.lower() == anime_name.lower()].index

    if len(anime_indices) == 0:
        print("Anime not found in the dataset.")
        return

    anime_index = anime_indices[0]

    # Get similarity scores
    similarity_scores = cosine_sim[anime_index]

    # Get indices above the threshold
    similar_indices = np.where(similarity_scores >= threshold)[0]

    # Remove the selected anime
    similar_indices = similar_indices[similar_indices != anime_index]

    # Sort by similarity score
    similar_indices = similar_indices[
        np.argsort(similarity_scores[similar_indices])[::-1]
    ]

    # Limit recommendations
    similar_indices = similar_indices[:max_recommendations]

    recommendations = df.iloc[similar_indices][
        ['name', 'genre', 'type', 'episodes', 'rating', 'members']
    ].copy()

    recommendations['Similarity Score'] = similarity_scores[similar_indices]

    return recommendations.reset_index(drop=True)

In [15]:
print("Threshold = 0.3")
display(recommend_by_threshold("Naruto", threshold=0.3, max_recommendations=10))

Threshold = 0.3


,name,genre,type,episodes,rating,members,Similarity Score
0,Fairy Tail,"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,175.0,8.22,584590,0.993032
1,D.Gray-man,"Action, Adventure, Comedy, Shounen",TV,103.0,8.20,334399,0.974623
2,Dragon Ball GT,"Action, Adventure, Comedy, Fantasy, Magic, Sci...",TV,64.0,6.72,226625,0.974601
3,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",TV,148.0,9.13,425855,0.973229
4,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",TV,153.0,8.16,316102,0.972025
5,Bleach,"Action, Comedy, Shounen, Super Power, Supernat...",TV,366.0,7.95,624055,0.967331
6,Soul Eater,"Action, Adventure, Comedy, Fantasy, Shounen, S...",TV,51.0,8.08,580184,0.957075
7,Fullmetal Alchemist,"Action, Adventure, Comedy, Drama, Fantasy, Mag...",TV,51.0,8.33,600384,0.955219
8,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,0.954936
9,Fairy Tail (2014),"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,102.0,8.25,255076,0.949190


In [16]:
print("Threshold = 0.5")
display(recommend_by_threshold("Naruto", threshold=0.5, max_recommendations=10))

Threshold = 0.5


,name,genre,type,episodes,rating,members,Similarity Score
0,Fairy Tail,"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,175.0,8.22,584590,0.993032
1,D.Gray-man,"Action, Adventure, Comedy, Shounen",TV,103.0,8.20,334399,0.974623
2,Dragon Ball GT,"Action, Adventure, Comedy, Fantasy, Magic, Sci...",TV,64.0,6.72,226625,0.974601
3,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",TV,148.0,9.13,425855,0.973229
4,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",TV,153.0,8.16,316102,0.972025
5,Bleach,"Action, Comedy, Shounen, Super Power, Supernat...",TV,366.0,7.95,624055,0.967331
6,Soul Eater,"Action, Adventure, Comedy, Fantasy, Shounen, S...",TV,51.0,8.08,580184,0.957075
7,Fullmetal Alchemist,"Action, Adventure, Comedy, Drama, Fantasy, Mag...",TV,51.0,8.33,600384,0.955219
8,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,0.954936
9,Fairy Tail (2014),"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,102.0,8.25,255076,0.949190


In [17]:
print("Threshold = 0.7")
display(recommend_by_threshold("Naruto", threshold=0.7, max_recommendations=10))

Threshold = 0.7


,name,genre,type,episodes,rating,members,Similarity Score
0,Fairy Tail,"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,175.0,8.22,584590,0.993032
1,D.Gray-man,"Action, Adventure, Comedy, Shounen",TV,103.0,8.20,334399,0.974623
2,Dragon Ball GT,"Action, Adventure, Comedy, Fantasy, Magic, Sci...",TV,64.0,6.72,226625,0.974601
3,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",TV,148.0,9.13,425855,0.973229
4,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",TV,153.0,8.16,316102,0.972025
5,Bleach,"Action, Comedy, Shounen, Super Power, Supernat...",TV,366.0,7.95,624055,0.967331
6,Soul Eater,"Action, Adventure, Comedy, Fantasy, Shounen, S...",TV,51.0,8.08,580184,0.957075
7,Fullmetal Alchemist,"Action, Adventure, Comedy, Drama, Fantasy, Mag...",TV,51.0,8.33,600384,0.955219
8,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,0.954936
9,Fairy Tail (2014),"Action, Adventure, Comedy, Fantasy, Magic, Sho...",TV,102.0,8.25,255076,0.949190


In [18]:
thresholds = [0.3, 0.5, 0.7]

print("Recommendation Count by Similarity Threshold")
print("-" * 50)

for threshold in thresholds:
    recommendations = recommend_by_threshold(
        "Naruto",
        threshold=threshold,
        max_recommendations=100
    )

    if recommendations is not None:
        print(
            f"Threshold {threshold}: "
            f"{len(recommendations)} recommendations"
        )

Recommendation Count by Similarity Threshold
--------------------------------------------------
Threshold 0.3: 100 recommendations
Threshold 0.5: 100 recommendations
Threshold 0.7: 100 recommendations


In [19]:
print("\nAverage Similarity by Threshold")
print("-" * 50)

for threshold in thresholds:
    recommendations = recommend_by_threshold(
        "Naruto",
        threshold=threshold,
        max_recommendations=100
    )

    if recommendations is not None and len(recommendations) > 0:
        avg_score = recommendations['Similarity Score'].mean()
        print(f"Threshold {threshold}: {avg_score:.4f}")


Average Similarity by Threshold
--------------------------------------------------
Threshold 0.3: 0.9321
Threshold 0.5: 0.9321
Threshold 0.7: 0.9321


In [20]:
recommend_anime("One Piece", 10)

,name,genre,type,episodes,rating,members,Similarity Score
0,One Punch Man,"Action, Comedy, Parody, Sci-Fi, Seinen, Super ...",TV,12.0,8.82,552458,0.994499
1,Code Geass: Hangyaku no Lelouch R2,"Action, Drama, Mecha, Military, Sci-Fi, Super ...",TV,25.0,8.98,572888,0.994485
2,Kill la Kill,"Action, Comedy, School, Super Power",TV,24.0,8.23,508118,0.994339
3,Guilty Crown,"Action, Drama, Sci-Fi, Super Power",TV,22.0,7.81,460959,0.993149
4,Code Geass: Hangyaku no Lelouch,"Action, Mecha, Military, School, Sci-Fi, Super...",TV,25.0,8.83,715151,0.993099
5,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,2.0,7.94,533578,0.992752
6,Darker than Black: Kuro no Keiyakusha,"Action, Mystery, Sci-Fi, Super Power",TV,25.0,8.25,440334,0.992470
7,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power",TV,25.0,8.54,896229,0.992297
8,Noragami,"Action, Adventure, Shounen, Supernatural",TV,12.0,8.17,515378,0.991868
9,Tengen Toppa Gurren Lagann,"Action, Adventure, Comedy, Mecha, Sci-Fi",TV,27.0,8.78,562962,0.991128


In [21]:
df[df['name'].str.contains("One Piece", case=False, na=False)][
    ['name', 'genre', 'type']
].head(10)

,name,genre,type
74,One Piece,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",TV
143,One Piece Film: Strong World,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",Movie
163,One Piece Film: Z,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",Movie
212,One Piece Film: Gold,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",Movie
231,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",Special
241,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",Special
352,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",OVA
430,One Piece 3D2Y: Ace no shi wo Koete! Luffy Nak...,"Adventure, Comedy, Fantasy, Shounen",Special
753,One Piece: Episode of Luffy - Hand Island no B...,"Action, Adventure, Comedy, Fantasy, Shounen, S...",Special
896,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",Special


## Performance Analysis

The anime recommendation system was implemented using cosine similarity. The anime genres were converted into numerical features using TF-IDF, while rating, number of episodes, and number of members were standardized before combining them with the genre features.

Cosine similarity was then calculated between the feature vectors of all anime. A recommendation function was developed that accepts an anime title and returns the most similar anime based on their similarity scores.

Different similarity thresholds were tested:

* **0.3:** Produces a larger recommendation list, but some recommendations may have relatively weaker similarity.
* **0.5:** Provides a balanced set of recommendations with moderate similarity.
* **0.7:** Produces fewer recommendations, but the recommended anime have stronger similarity to the selected anime.

### Areas for Improvement

The recommendation system can be improved in several ways:

1. Include additional anime attributes such as type and popularity.
2. Give different weights to genres, ratings, episodes, and members.
3. Use user-rating information to develop a collaborative filtering model.
4. Handle anime with missing or incomplete genre information more effectively.
5. Remove extremely rare or duplicate anime titles.
6. Use a hybrid recommendation system combining content-based and collaborative filtering techniques.

## Conclusion

The cosine similarity approach successfully provides anime recommendations based on similarities between anime features. The system demonstrates how categorical information such as genres can be transformed using TF-IDF and combined with normalized numerical features.

The similarity threshold allows the recommendation system to control the trade-off between the number and quality of recommendations. Lower thresholds provide more recommendations, while higher thresholds provide fewer but more closely related recommendations.

Overall, the project demonstrates the application of feature extraction, normalization, cosine similarity, and recommendation techniques to build a basic content-based anime recommendation system.
